# 

This notebook uses the project's actual `Indexer` class and preprocessing method (from `deep-regex-model/preprocess.py`) to build vocabularies and encode data properly, then integrates with Hugging Face.

In [3]:
# Run preprocessing on all 3 datasets from endsem/datasets and merge them
import subprocess
import sys

DATASET_DIRS = "datasets/KB13,datasets/NL-RX-Synth,datasets/NL-RX-Turk"
OUTPUT_DIR = "datasets/splits"

print("Running preprocessing on all 3 datasets from endsem/datasets...")
ret = subprocess.call([
    sys.executable,
    "preprocess_data.py",
    "--data_dirs",
    DATASET_DIRS,
    "--output_dir",
    OUTPUT_DIR,
    "--targ_separate",
    "1",
])
print("preprocess exit code:", ret)
print(f"Merged splits saved to {OUTPUT_DIR}/")

Running preprocessing on all 3 datasets from endsem/datasets...
preprocess exit code: 0
Merged splits saved to datasets/splits/


In [1]:
# Build vocabularies using local copy of project Indexer logic
import os
import pickle
import pandas as pd
from indexer_local import Indexer

splits_dir = "datasets/splits"
train_csv = os.path.join(splits_dir, "train.csv")
src_train_txt = os.path.join(splits_dir, "src-train.txt")
targ_train_txt = os.path.join(splits_dir, "targ-train.txt")

# Support both formats:
# 1) train.csv with columns nl/regex
# 2) src-train.txt + targ-train.txt produced by preprocess_data.py
if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)
elif os.path.exists(src_train_txt) and os.path.exists(targ_train_txt):
    with open(src_train_txt, "r", encoding="utf-8") as f:
        src_lines = [line.rstrip("\n") for line in f]
    with open(targ_train_txt, "r", encoding="utf-8") as f:
        targ_lines = [line.rstrip("\n") for line in f]
    if len(src_lines) != len(targ_lines):
        raise ValueError(f"Split size mismatch: src={len(src_lines)} targ={len(targ_lines)}")
    train_df = pd.DataFrame({"nl": src_lines, "regex": targ_lines})
else:
    raise FileNotFoundError(
        "No train split found. Expected either datasets/splits/train.csv or "
        "datasets/splits/src-train.txt + datasets/splits/targ-train.txt"
    )

# Match project special tokens
src_indexer = Indexer(["<blank>", "<unk>", "<s>", "</s>"])
target_indexer = Indexer(["<blank>", "<unk>", "<s>", "</s>"])
char_indexer = Indexer(["<blank>", "<unk>", "{", "}"])
char_indexer.add_w([src_indexer.PAD, src_indexer.UNK, src_indexer.BOS, src_indexer.EOS])

for _, row in train_df.iterrows():
    src_words = str(row["nl"]).strip().split()
    targ_words = str(row["regex"]).strip().split()

    for word in src_words:
        src_indexer.vocab[word] += 1
    for word in targ_words:
        target_indexer.vocab[word] += 1

    for word in src_words + targ_words:
        clean_word = char_indexer.clean(word)
        if len(clean_word) == 0:
            continue
        for ch in list(clean_word):
            char_indexer.vocab[ch] += 1

# Keep full vocab like behavior when no prune threshold is provided
src_indexer.prune_vocab(len(src_indexer.vocab))
target_indexer.prune_vocab(len(target_indexer.vocab))
char_indexer.prune_vocab(len(char_indexer.vocab))

with open("indexers.pkl", "wb") as f:
    pickle.dump({"src": src_indexer, "target": target_indexer, "char": char_indexer}, f)

print(
    f"Loaded {len(train_df)} train pairs | saved indexers.pkl | "
    f"src_vocab={len(src_indexer.d)} target_vocab={len(target_indexer.d)} char_vocab={len(char_indexer.d)}"
)

Loaded 13535 train pairs | saved indexers.pkl | src_vocab=525 target_vocab=68 char_vocab=82


In [5]:
# Load local indexers and encode sequences 
import pickle

with open("indexers.pkl", "rb") as f:
    indexers = pickle.load(f)
src_indexer = indexers["src"]
target_indexer = indexers["target"]
char_indexer = indexers["char"]

MAX_SEQ_LEN = 100
MAX_WORD_LEN = 20


def pad(ls, length, symbol):
    if len(ls) >= length:
        return ls[:length]
    return ls + [symbol] * (length - len(ls))


def encode_sequence(seq_words, indexer, use_chars=False):
    seq = [indexer.BOS] + seq_words + [indexer.EOS]
    seq = pad(seq, MAX_SEQ_LEN, indexer.PAD)
    seq_ids = indexer.convert_sequence(seq)

    seq_chars = None
    if use_chars:
        seq_chars = []
        for word in seq:
            clean_word = char_indexer.clean(word)
            char_seq = [char_indexer.BOS] + list(clean_word) + [char_indexer.EOS]
            char_seq = pad(char_seq, MAX_WORD_LEN, char_indexer.PAD)
            char_ids = char_indexer.convert_sequence(char_seq)
            seq_chars.append(char_ids)

    return seq_ids, seq_chars


def tokenize_batch_project(batch, use_chars=False):
    src_ids_list = []
    targ_ids_list = []
    src_chars_list = [] if use_chars else None
    targ_chars_list = [] if use_chars else None

    for nl, regex in zip(batch["nl"], batch["regex"]):
        src_words = str(nl).strip().split()
        targ_words = str(regex).strip().split()

        src_ids, src_chars = encode_sequence(src_words, src_indexer, use_chars)
        targ_ids, targ_chars = encode_sequence(targ_words, target_indexer, use_chars)

        src_ids_list.append(src_ids)
        targ_ids_list.append(targ_ids)
        if use_chars:
            src_chars_list.append(src_chars)
            targ_chars_list.append(targ_chars)

    out = {
        "input_ids": src_ids_list,
        "labels": targ_ids_list,
    }
    if use_chars:
        out["src_char_ids"] = src_chars_list
        out["targ_char_ids"] = targ_chars_list
    return out


print("Preprocessing functions ready")

Preprocessing functions ready
